In [ ]:
from sklearn.datasets import fetch_openml
import joblib
import os
if os.path.exists("mnist_dataset.joblib"):
    print("loading from device")
    mnist = joblib.load("mnist_dataset.joblib")
else:
    print("fetching from server")
    mnist = fetch_openml("mnist_784", version=1)
    joblib.dump(mnist, "mnist_dataset.joblib")
mnist.keys()

In [ ]:
print(mnist["DESCR"])

In [ ]:
X, y = mnist.data, mnist.target

In [ ]:
X.shape, y.shape

In [ ]:
X

In [ ]:
firstImage = X.iloc[0,:].to_numpy().reshape(28, 28)
firstImage[0]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.imshow(firstImage, cmap=cm.binary)
plt.axis("off")
print("label: ",y[0], type(y[0]))

In [ ]:
import numpy as np
y=y.astype(np.float64)
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

In [ ]:
from sklearn.linear_model import SGDClassifier
#  doing binay classififcation for 5 or (not 5)
y_train_5, y_test_5 = y_train==5, y_test==5
print(y_train_5.value_counts())
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)

In [ ]:
sgd_clf.predict([X.iloc[0, :]])

In [ ]:
from sklearn.model_selection import cross_val_score
a123 = cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")
b123 = cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="precision")
c123 = cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="recall")
print("accuracy: ", a123, "\nprecesion: ", b123, "\nRecall:", c123)

In [ ]:
from sklearn.model_selection import cross_val_predict
y_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

In [ ]:
y_pred.shape

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train_5, y_pred)

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score
print("precision_score: ", precision_score(y_train_5, y_pred))
print("recall_score: ", recall_score(y_train_5, y_pred))
print("accracy_score: ", accuracy_score(y_train_5, y_pred))


In [ ]:
from sklearn.metrics import f1_score
print("f1_score: ", f1_score(y_train_5, y_pred))

In [ ]:
y_scores = sgd_clf.decision_function([X.iloc[0, :]])
y_scores

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3, method="decision_function")

In [ ]:
from sklearn.metrics import precision_recall_curve
prec, rec, thres = precision_recall_curve(y_train_5, y_scores)

In [ ]:
def plot_prec_recall_vs_thres(prec, rec, thres):
    plt.plot(thres, prec[:-1],"b--", label="precesion")
    plt.plot(thres, rec[:-1],"g-" ,label="recall")
    plt.legend()
plot_prec_recall_vs_thres(prec, rec, thres)

In [ ]:
plt.plot(rec, prec)

In [ ]:
thres_90_prec = thres[np.argmax(prec>0.9)]

In [ ]:
y_train_pred_90 = (y_scores>=thres_90_prec)

In [ ]:
precision_score(y_train_5, y_train_pred_90)

In [ ]:
recall_score(y_train_5, y_train_pred_90)

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thr = roc_curve(y_train_5, y_scores)
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0,1],[0,1],"k--")
plot_roc_curve(fpr, tpr)

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_train_5, y_scores)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(random_state=42)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3, method="predict_proba")

In [ ]:
y_scores_forest = y_probas_forest[:, 1]
fpr_forest, tpr_forest, thres_forest = roc_curve(y_train_5, y_scores_forest)

In [ ]:
plot_roc_curve(fpr_forest, tpr_forest, "Random Forest")
plt.plot(fpr, tpr, "b:", label="SGD")
plt.legend()

In [ ]:
roc_auc_score(y_train_5, y_scores_forest)

In [ ]:
# y_scores_forest.shape, y_train_5.shape
# y_scores_forest[:10]
precision_score(y_train_5, y_scores_forest>0.5), recall_score(y_train_5, y_scores_forest>0.5)

In [ ]:
sgd_clf.fit(X_train, y_train)
sgd_clf.predict([X.iloc[0, :]])

In [ ]:
sgd_clf.classes_

In [ ]:
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")

In [ ]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
x_train_scaled = scalar.fit_transform(X_train.astype(np.float64))
cross_val_score(sgd_clf, x_train_scaled, y_train, cv=3, scoring="accuracy")

In [ ]:
y_train_pred = cross_val_predict(sgd_clf, x_train_scaled, y_train, cv=3)
conf_mat = confusion_matrix(y_train, y_train_pred)
conf_mat

In [ ]:
plt.matshow(conf_mat, cmap=plt.cm.gray)
plt.show()

In [ ]:
row_sums = conf_mat.sum(axis=1, keepdims = True)
norm_conf_mat = conf_mat/row_sums

In [ ]:
np.fill_diagonal(norm_conf_mat, 0)
plt.matshow(norm_conf_mat, cmap=plt.cm.gray)
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knei = KNeighborsClassifier()
knei.fit(X_train, y_train)

In [ ]:
# knei.predict(X_train.iloc[:5,:].to_numpy())
accuracy_score(y_train, knei.predict(X_train.to_numpy()))

In [ ]:
accuracy_score(y_test.to_numpy(), knei.predict(X_test.to_numpy()))

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid=[{"n_neighbors":[1,3,5,7], "weights":["uniform","distance"]}]
grid_search = GridSearchCV(
    knei,param_grid, cv=5, scoring="accuracy"
)

In [ ]:
grid_search.fit(X_train.to_numpy(), y_train.to_numpy())

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_estimator_

In [ ]:
import pandas as pd
resul = pd.DataFrame(grid_search.cv_results_)
print(resul)

In [ ]:
model = grid_search.best_estimator_
accuracy_score(y_test.to_numpy(), model.predict(X_test.to_numpy()))